# CW_06
# Beam Propagation in Waveguides

In this exercise you will investigate how we can use the beam propagation method to investigate the modes of a waveguide. 

To keep computation simple, you will use the 1D beam propagation method for the whole exercise.

In [ ]:
# - No modification necessary -

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve
from scipy.fft import fft, ifft, fftfreq
from ipywidgets import interact, FloatSlider

# -----------------------
# Global simulation params
# -----------------------
wavelength = 800e-9
k0 = 2 * np.pi / wavelength

# Spatial grid
Nx = 2048
Nz = 500
x_max = 15e-6
z_max = 100e-6

x = np.linspace(-x_max, x_max, Nx)
z = np.linspace(0, z_max, Nz)

dx = x[1] - x[0]
dz = z[1] - z[0]

# Refractive indices
n_core = 1.5
n_clad = 1.45

# Part 1

In this part, you will create the waveguide that we are going to propagate through. The waveguide should be defined as a 2D array in x and z so that later you can investigate what happens as the waveguide changes along its length.

Implement the function create_slab_waveguide based on the provided parameters. and use the provided plotting code to visualize the waveguide.

In [ ]:
def create_slab_waveguide(width, n_core, n_clad, z_profile=None):
    raise NotImplementedError


def plot_waveguide(n):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    
    # Cross-section
    ax[0].plot(x * 1e6, n[0])
    ax[0].set_title("Waveguide Cross-section")
    ax[0].set_xlabel("x (µm)")
    ax[0].set_ylabel("Refractive Index")
    
    # 2D view
    im = ax[1].imshow(n.T,
                  extent=[z[0]*1e6, z[-1]*1e6,
                          x[0]*1e6, x[-1]*1e6],
                  aspect='auto',
                  origin='lower',
                  cmap='viridis')

    ax[1].set_xlabel("z (µm)")
    ax[1].set_ylabel("x (µm)")
    ax[1].set_title("Waveguide (z-x)")
    
    plt.colorbar(im, ax=ax[1])
    plt.tight_layout()
    plt.show()

In [ ]:
# - No modification necessary -

width = 2e-6
n = create_slab_waveguide(width, n_core, n_clad)
plot_waveguide(n)

# Part 2

In this part you will be responsible for analytically solving for the fundamental mode (TE) of the slab waveguide. 

To do this, you will first need to solve for the propagation constant of the mode. To do this, think to a ray optics perspective of how the light travels within the waveguide. Remember that after each cycle of bouncing and propagating, a phase shift of 2pi must be accumulated so that the wavefronts constructively interfere. The propagation constant represents the speed of propagation of the mode, so it can be calculated from how much of the wave is propagating along the axis of the waveguide. Computationally, solving this will entail setting up the equation that represents this condition, manipulating it to an equation that can be set to 0, and then using the scipy fsolve function to get the propagation constant that satisfies the specified equilibrium.

For more information about calculating the modes of a slab waveguide, refer to this source: https://engineering.purdue.edu/wcchew/ece604s21/Lecture%20Notes/Lect17.pdf

After you have calculated the propagation constant of the mode, you will have to use this to create the actual profile of the mode. To do this, recall that within the wave will take the form of being cosinusoidal within the core and exponentially decaying in the cladding. 

Within the core the frequency constant will be $\sqrt{(n_{core} k_0)^2 - \beta^2}$ and within the cladding the exponent will be $\sqrt{\beta^2 - (n_{clad} k_0)^2}$.

After you have solved for the fundamental mode, use the provided plotting code to show the resulting curve.


In [ ]:
def slab_dispersion_TE(beta, width, n_core, n_clad):
    raise NotImplementedError

def solve_beta(width, n_core, n_clad):
    raise NotImplementedError

In [ ]:
def compute_mode_profile(beta, width, n_core, n_clad):
    raise NotImplementedError

In [ ]:
# - No modification necessary -

def plot_modes(n, width):
    beta_TE = solve_beta(width, n_core, n_clad)

    mode_TE = compute_mode_profile(beta_TE, width, n_core, n_clad)

    plt.figure(figsize=(6,4))
    plt.plot(x*1e6, n[0]/np.max(n[0]), 'k--', label="Waveguide")
    plt.plot(x*1e6, mode_TE, label="TE Mode")
    
    plt.xlabel("x (µm)")
    plt.title("Fundamental Mode")
    plt.legend()
    plt.show()
    
    return mode_TE

In [ ]:
mode_TE = plot_modes(n, width)

# Part 3

Now that you have solved for the fundamental mode, let's investigate how it propagates through the waveguide and how this differs from the behavior of propagating a gaussian beam.

In this part, you will just need to implement the function gaussian_beam to generate a gaussian input of the provided FWHM (remember that FWHM is not the same as standard deviation).

Then use the provided plotting code to see that the differences in propagation.

In [ ]:
# - No modification necessary -

def bpm_propagate(field0, n, n_ref):
    field = field0.copy()
    result = np.zeros((Nz, Nx), dtype=complex)
    
    kx = 2 * np.pi * fftfreq(Nx, d=dx)
    
    for i in range(Nz):
        result[i] = field
        
        # Diffraction (Fourier step)
        F = fft(field)
        F *= np.exp(-1j * (kx**2) * dz / (2 * k0 * n_ref))
        field = ifft(F)
        
        # Refraction
        phase = np.exp(1j * k0 * (n[i] - n_ref) * dz)
        field *= phase
    
    return result

In [ ]:
def gaussian_beam(width):
    raise NotImplementedError

In [ ]:
# - No modification necessary -

def plot_propagation(field, title=""):
    intensity = np.abs(field)**2
    
    plt.figure(figsize=(8,4))
    
    plt.imshow(intensity.T,   # <-- transpose!
               extent=[z[0]*1e6, z[-1]*1e6,   # x-axis = z
                       x[0]*1e6, x[-1]*1e6],  # y-axis = x
               aspect='auto',
               origin='lower',
               cmap='inferno')
    
    plt.xlabel("z (µm)")
    plt.ylabel("x (µm)")
    plt.title(title)
    plt.colorbar(label="Intensity")
    plt.show()

In [ ]:
# - No modification necessary -

# Fundamental mode propagation
field_TE = bpm_propagate(mode_TE, n, n_clad)
plot_propagation(field_TE, "TE Mode Propagation")

# Gaussian propagation
gauss = gaussian_beam(width)
field_gauss = bpm_propagate(gauss, n, n_clad)
plot_propagation(field_gauss, "Gaussian Propagation")

# Part 4

The last componenet of this workbook will be to look at how the core and cladding refractive indices affect the numerical aperture of the waveguide.

To do this, you will need to implement the functions numerical_aperture and divergence_angle. The function truncated_waveguide has already been provided for you, and will generate a waveguide that stops before the end of the simulation so that you can see what happens when the light exits the waveguide and propagates in free space.

Once you have implemented these functions, take some time to play around with the interactive plot that has been provided and make sure that what you are seeing matches your expectations for the system.

In [ ]:
# - No modification necessary -

def truncated_waveguide(width, n_core, n_clad, cutoff_z):
    def z_profile(z_val):
        return width if z_val < cutoff_z else 0
    return create_slab_waveguide(width, n_core, n_clad, z_profile)

In [ ]:
def numerical_aperture(n_core, n_clad):
    raise NotImplementedError

def divergence_angle(n_core, n_clad):
    raise NotImplementedError


In [ ]:
# - No modification necessary -

def simulate_truncated(width, n_core, n_clad):
    n_trunc = truncated_waveguide(width, n_core, n_clad, z_max/2)
    
    beta_TE = solve_beta(width, n_core, n_clad)
    mode_TE = compute_mode_profile(beta_TE, width, n_core, n_clad)
    
    field = bpm_propagate(mode_TE, n_trunc, n_clad)
    
    plot_propagation(field, f"Truncated Waveguide (width={width*1e6:.2f} µm)")
    
    print("Numerical Aperture:", numerical_aperture(n_core, n_clad))
    print("Divergence angle (rad):", divergence_angle(n_core, n_clad))

interact(simulate_truncated,
             width=FloatSlider(min=0.2e-6, max=2.4e-6, step=0.2e-6, readout_format='.6f'),
             n_core=FloatSlider(min=1.4, max=1.7, step=0.01, value=1.6, readout_format='.3f'),
             n_clad=FloatSlider(min=1.4, max=1.7, step=0.01, value=1.5, readout_format='.3f'),
         )
        